# LLM Evaluation - Qwen Medical LoRA
Framework based on ubuntu_evaluation.py adapted for Jupyter notebook output

In [ ]:
import json
import torch
from pathlib import Path
from datetime import datetime
from typing import Dict, List
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

print("="*80)
print("LLM EVALUATION - QWEN MEDICAL LORA")
print("="*80)

## 1. Configuration & Load Test Cases

In [ ]:
# Paths
QWEN_MODEL_PATH = 'Qwen/Qwen2.5-7B'
QWEN_LORA_PATH = 'qwen_medical_lora_gpu'
TEST_DIRS = [Path('json samples'), Path('multiple shunts in 1 sesh')]

# CHIVA Rules
CHIVA_RULES = """
=== CHIVA VENOUS SHUNT CLASSIFICATION RULES ===

ANATOMY:
    N1 = Deep venous system (femoral/popliteal vein)
    N2 = Great Saphenous Vein (GSV) or Small Saphenous Vein (SSV) trunk
    N3 = Tributaries / superficial branches
    EP = Physiological (forward, antegrade) flow
    RP = Retrograde (pathological, reflux) flow

CRITICAL RULE — SFJ COMPETENCE:
    SFJ is INCOMPETENT if and only if a clip has fromType=N1 AND toType=N2 (EP N1→N2).
    EP N2→N2 means blood circulates within the saphenous trunk via a perforator — SFJ REMAINS COMPETENT.

CLASSIFICATION:
    STEP 1: Check for EP N1→N2
        YES → SFJ INCOMPETENT (Case A/B)
        NO → SFJ COMPETENT (Case C)

    Case A (EP N1→N2, NO EP N2→N3):
        RP N2→N1, no RP at N3 → TYPE 1

    Case B (EP N1→N2 AND EP N2→N3):
        RP N3 only, NO RP N2→N1 → TYPE 3
        RP N3 AND RP N2→N1, elim=\"Reflux\" → TYPE 1+2
        RP N3 AND RP N2→N1, elim=\"No Reflux\" → TYPE 3

    Case C (NO EP N1→N2):
        EP N2→N3 → TYPE 2A
        EP N2→N2, RP N3, NO RP N2→N1 → TYPE 2B
        EP N2→N2, RP N3, RP N2→N1 → TYPE 2C
"""

FEW_SHOT_LIGATION = """
LIGATION PLANNING EXAMPLES:

Example 1 (TYPE 1):
    Ligate at saphenofemoral junction. Preserve distal GSV if quality permits.

Example 2 (TYPE 2A):
    Selective perforator ligation with duplex guidance at N2→N3 junction.

Example 3 (TYPE 2C):
    Ligate perforator entry point AND all GSV reflux sites along trunk.

Example 4 (TYPE 3):
    Ligate tributary at N2→N3 junction. Follow up 6-12 months; if GSV reflux develops, ligate SFJ.

Example 5 (TYPE 1+2):
    Combined approach - ligate SFJ AND refluxing tributaries simultaneously.
"""

print("✓ Configuration loaded")

In [ ]:
# Load test cases
print("\n[1/4] Loading test cases...")
test_cases = []

for test_dir in TEST_DIRS:
    if test_dir.exists():
        json_files = list(test_dir.glob("**/*.json"))
        for json_file in sorted(json_files):
            try:
                with open(json_file, 'r', encoding='utf-8', errors='ignore') as f:
                    data = json.load(f)
                    clips = data.get('clips', [])
                    if clips:
                        test_cases.append({
                            'name': json_file.stem,
                            'clips': clips,
                            'source': test_dir.name,
                        })
            except Exception as e:
                pass

print(f"  Loaded {len(test_cases)} test cases")
for tc in test_cases[:3]:  # Show first 3
    print(f"    - {tc['name']} ({tc['source']})")

## 2. Load Model

In [ ]:
print("\n[2/4] Loading Qwen Medical LoRA...")

# Load base model
print("  Loading base model...", end=" ", flush=True)
base_model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL_PATH,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
print("OK")

# Load LoRA
print("  Loading LoRA adapters...", end=" ", flush=True)
model = PeftModel.from_pretrained(base_model, QWEN_LORA_PATH, is_trainable=False)
tokenizer = AutoTokenizer.from_pretrained(QWEN_LORA_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model.eval()
device = next(model.parameters()).device
print(f"OK (on {device})")

print(f"\n✓ Model loaded: {model.config.model_type} with LoRA")

## 3. Helper Functions

In [ ]:
def _clip_label(flow: str, ft: str, tt: str, y: float) -> str:
    """Annotate clip with anatomical significance"""
    if flow == "EP" and ft == "N1" and tt == "N2":
        if y <= 0.098:
            return " [SFJ-ENTRY=INCOMPETENT]"
        return " [Hunterian-ENTRY=INCOMPETENT]" if y <= 0.353 else " [Deep-to-GSV-ENTRY]"
    if flow == "RP" and ft == "N3":
        return f" [TRIBUTARY-REFLUX: N3→{tt}]"

    labels = {
        ("EP", "N2", "N2"): " [PERFORATOR-ENTRY: N2→N2, SFJ=COMPETENT]",
        ("EP", "N2", "N3"): " [GSV-to-TRIBUTARY-ENTRY: N2→N3]",
        ("RP", "N2", "N1"): " [GSV-TRUNK-REFLUX: N2→N1]",
    }
    return labels.get((flow, ft, tt), "")

def _summarise_clips(clips: List[Dict]) -> str:
    """Format clips with anatomical labels"""
    lines = []
    for i, c in enumerate(clips):
        flow = c.get('flow', '?')
        ft = c.get('fromType', '?')
        tt = c.get('toType', '?')
        y = c.get('posYRatio') or 0.0
        loc = _clip_label(flow, ft, tt, y)
        lines.append(f"  Clip {i:02d}: {flow} {ft}→{tt}  y={y:.3f}{loc}")
    return "\n".join(lines)

def format_clips_v1(clips: List[Dict]) -> str:
    """V1: Clip notation"""
    descriptions = []
    for clip in clips:
        flow = "EP" if clip.get('flow') == 'EP' else "RP"
        from_to = f"{clip.get('fromType')}->{clip.get('toType')}"
        y_val = clip.get('posYRatio', 0)
        descriptions.append(f"{flow} {from_to} (y={y_val:.3f})")
    return "Duplex findings: " + ", ".join(descriptions)

def format_clips_v2(clips: List[Dict]) -> str:
    """V2: Medical terminology"""
    findings = []
    for clip in clips:
        flow = clip.get('flow')
        from_type = clip.get('fromType')
        to_type = clip.get('toType')

        if flow == 'EP' and from_type == 'N1' and to_type == 'N2':
            findings.append("antegrade flow from deep femoral vein to saphenous trunk indicating saphenofemoral junction incompetence")
        elif flow == 'EP' and from_type == 'N2' and to_type == 'N3':
            findings.append("antegrade flow from saphenous trunk feeding into tributaries")
        elif flow == 'RP' and from_type == 'N2' and to_type == 'N1':
            findings.append("retrograde reflux within saphenous trunk toward deep system")
        elif flow == 'RP' and from_type == 'N3' and to_type == 'N2':
            findings.append("retrograde reflux in tributary branches toward saphenous trunk")
        elif flow == 'RP' and from_type == 'N3' and to_type == 'N1':
            findings.append("retrograde reflux in tributary branches toward deep venous system")

    return "Duplex ultrasound demonstrates: " + "; ".join(findings) + " in patient with chronic venous insufficiency."

def extract_shunt_type(text: str) -> str:
    """Extract type from response"""
    import re
    try:
        data = json.loads(text)
        result = data.get('type') or data.get('shunt_type')
        if result:
            return result.strip()
    except:
        match = re.search(r'TYPE\s+[\d+A-Z]+|No\s+shunt', text, re.IGNORECASE)
        if match:
            return match.group(0).upper()
    return text.strip()[:50] if text else "UNKNOWN"

print("✓ Helper functions defined")

## 4. Inference Functions

In [ ]:
def build_classification_prompt(clips_summary: str) -> str:
    """Build structured classification prompt"""
    return f"""{CHIVA_RULES}

=== CLINICAL ASSESSMENT ===
{clips_summary}

═══════════════════════════════════════════════════════════════
DECISION GUIDE
═══════════════════════════════════════════════════════════════

STEP 1: CHECK FOR EP N1→N2 (SFJ or Hunterian ENTRY)
    Look for: "EP N1→N2" with [SFJ-ENTRY=INCOMPETENT] or [Hunterian-ENTRY=INCOMPETENT] label
    If YES → SFJ INCOMPETENT (go to Case B or A)
    If NO  → SFJ COMPETENT (go to Case C)

STEP 2: CHECK FOR REFLUX PATTERNS
    2a) ANY RP N3→N2 or RP N3→N1? (tributary reflux)
    2b) ANY RP N2→N1? (GSV trunk reflux)
    2c) ANY EP N2→N3? (extra antegrade to tributary)

STEP 3: PATTERN MATCHING
    SFJ INCOMPETENT (has EP N1→N2):
    - NO EP N2→N3 + RP N2→N1 = TYPE 1
    - YES EP N2→N3 + RP N3 only = TYPE 3
    - YES EP N2→N3 + RP N3 AND RP N2→N1 = TYPE 1+2

    SFJ COMPETENT (NO EP N1→N2):
    - EP N2→N3 EXISTS = TYPE 2A
    - EP N2→N2 + RP N3 only = TYPE 2B
    - EP N2→N2 + RP N3 AND RP N2→N1 = TYPE 2C

═══════════════════════════════════════════════════════════════

Output ONLY this JSON structure (no markdown, no explanation):

{{
    \"shunt_type\": \"TYPE 1 | TYPE 2A | TYPE 2B | TYPE 2C | TYPE 3 | TYPE 1+2 | No shunt detected\",
    \"confidence\": 0.85,
    \"reasoning\": \"your reasoning\",
    \"summary\": \"1 sentence\"
}}"""

def qwen_inference(prompt: str, task: str = "classification") -> str:
    """Qwen inference"""
    try:
        if task == "classification":
            messages = [
                {"role": "system", "content": "You are a CHIVA shunt classification expert. Output ONLY valid JSON with fields: type and reasoning. No markdown, no explanation outside JSON."},
                {"role": "user", "content": prompt}
            ]
            max_tok = 200
        else:
            messages = [
                {"role": "system", "content": f"""Expert vascular surgeon. Provide BRIEF ligation planning.
Output ONLY valid JSON with two sections: ligation_steps (max 4 lines) and ligation_reasoning (1 sentence).
Be concise and to the point.\n\n{CHIVA_RULES}\n\n{FEW_SHOT_LIGATION}"""},
                {"role": "user", "content": f"""{prompt}

Respond with ONLY this JSON format:
{{\"ligation_steps\": \"step 1, step 2, step 3, step 4 (max 4 lines)\", \"ligation_reasoning\": \"one sentence why\"}}"""}  
            ]
            max_tok = 300

        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(text, return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_tok,
                do_sample=False,
                temperature=0.0,
                pad_token_id=tokenizer.eos_token_id
            )

        result = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
        return result
    except Exception as e:
        return f"[Qwen Error: {str(e)[:80]}]"

print("✓ Inference functions defined")

## 5. Run Evaluation

In [ ]:
print("\n[3/4] Running evaluation...\n")

results = {
    'timestamp': datetime.now().isoformat(),
    'test_count': len(test_cases),
    'model': f'Qwen2.5-7B + {QWEN_LORA_PATH}',
    'evaluations': []
}

for idx, test in enumerate(test_cases[:5], 1):  # Limit to first 5 for notebook
    print(f"Test [{idx}/{min(5, len(test_cases))}]: {test['name']}")
    
    clips = test['clips']
    
    # Task 1: Classification
    clips_summary = _summarise_clips(clips)
    classification_prompt = build_classification_prompt(clips_summary)
    qwen_response = qwen_inference(classification_prompt, "classification")
    shunt_type = extract_shunt_type(qwen_response)
    
    # Task 2: Ligation
    v1_prompt = format_clips_v1(clips)
    ligation_prompt = f"{CHIVA_RULES}\n\n{FEW_SHOT_LIGATION}\n\nPatient presents with: {v1_prompt}\nClassified as: {shunt_type}\n\nProvide ligation planning."
    qwen_ligation = qwen_inference(ligation_prompt, "ligation")
    
    results['evaluations'].append({
        'test_name': test['name'],
        'source': test['source'],
        'clips_summary': clips_summary,
        'classification_response': qwen_response,
        'detected_type': shunt_type,
        'ligation_response': qwen_ligation,
    })
    
    print(f"  → Detected: {shunt_type}")
    print()

print("\n✓ Evaluation complete")

## 6. Display Results

In [ ]:
print("\n[4/4] Results Summary\n")
print("="*80)
print(f"Model: {results['model']}")
print(f"Timestamp: {results['timestamp']}")
print(f"Test Cases: {len(results['evaluations'])}")
print("="*80)

for result in results['evaluations']:
    print(f"\n{'='*80}")
    print(f"TEST: {result['test_name']} ({result['source']})")
    print(f"{'='*80}")
    
    print(f"\nCLIPS SUMMARY:")
    print(result['clips_summary'])
    
    print(f"\n{'─'*80}")
    print(f"CLASSIFICATION RESPONSE:")
    print(f"Detected Type: {result['detected_type']}")
    print(f"\nFull Response:")
    print(result['classification_response'][:500])  # First 500 chars
    
    print(f"\n{'─'*80}")
    print(f"LIGATION PLANNING:")
    print(result['ligation_response'][:500])  # First 500 chars

## 7. Save Results

In [ ]:
# Save results to JSON
with open('evaluation_notebook_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("\n✓ Results saved to evaluation_notebook_results.json")